In [ ]:
# Parameters (Papermill will override this)
RESULT_FOLDER_NAME = None

In [ ]:
# Set parent as root and import config
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import configs.simulation_config as cfg
from configs.paths import PROCESSED_DIR, DATA_DIR, RESULT_DIR

import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
from collections import defaultdict
from joblib import Parallel, delayed
import time

In [ ]:
def build_node_lookups(nodes):
    nodes = nodes.set_index("osmid")

    return {
        "transport_mode": nodes["transport_mode"],
        "population": nodes["pop_total"].fillna(0),
        "shelter_capacity": nodes["shelter_capacity"].fillna(1e6),
        "vehicle_occ": nodes["vehicle_occ"].fillna(1),
    }

In [ ]:
nodes = pd.read_csv(PROCESSED_DIR / "hatyai_nodes_transport.csv")
edges = pd.read_csv(PROCESSED_DIR / "hatyai_edges_with_dynamic_flood.csv")
edges_use = pd.read_csv(PROCESSED_DIR / "hatyai_edges_bidirection.csv")
lookups = build_node_lookups(nodes)

## Build time expanded graph

In [ ]:
timeline_df = pd.read_parquet(PROCESSED_DIR / "flood_simulation_timeline.parquet")
time_names = 't' + timeline_df['step_id'].astype(str)
time_stamps = timeline_df["timestamp"]
timestep_hour = [
    (time_stamps[i + 1] - time_stamps[i]).total_seconds() / 3600.0
    for i in range(len(time_stamps) - 1)
]

flood_cols_move = [f"flood_depth_{name}" for name in time_names[:-1]]
edges_min_cols = ["u", "v", "length", "capacity"] + flood_cols_move
edges_records = edges[edges_min_cols].to_records(index=False)

nodes_unique = nodes.drop_duplicates("osmid")
node_ids = nodes_unique["osmid"].astype(int).values

In [ ]:
def edge_speed_kmh(mode: str, flood_depth: float) -> float:
    if flood_depth >= cfg.IMPASSABLE_FLOOD_DEPTH: return 0.0
    base_speed = cfg.DEFAULT_MODE_SPEED_KMH.get(mode, 0.0)
    min_flood_depth = cfg.MIN_FLOOD_DEPTH.get(mode, 0.06)
    if flood_depth < min_flood_depth: return base_speed
    
    if mode == "walk":
        # Walking speed under flood formular from
        # https://doi.org/10.1016/j.ijdrr.2021.102192
        # This calculation assume floodwater speed = 0 (hydrostatic conditions)
        flood_force = (flood_depth ** 2) / 2
        calculated_speed_ms = 0.6629 * (flood_force ** -0.103)
        return calculated_speed_ms * 3.6 # converse m/s into km/h
    elif mode == "drive":
        # Vheicle speed under flood formular
        # https://doi.org/10.3390/w12040926
        calculated_speed_kmh = 85 * (cfg.E ** (-9 * flood_depth))
        return calculated_speed_kmh
    
    return base_speed


def travel_time_hours(length_m: float, speed_kmh: float) -> float:
    if speed_kmh <= 0: return np.inf
    return (length_m / 1000.0) / speed_kmh

def build_move_edges_for_time(ti):
    step_h = timestep_hour[ti]
    if step_h <= 0: return []

    flood_col = flood_cols_move[ti]
    out = []

    for rec in edges_records:
        u = int(rec.u)
        v = int(rec.v)

        pairs = [(u, v)] if not cfg.BIDIR_EVAC else [(u, v), (v, u)]

        flood_depth = getattr(rec, flood_col, 0)
        if pd.isna(flood_depth):
            flood_depth = 0

        for uu, vv in pairs:
            mode_u = lookups["transport_mode"].get(uu, "walk")

            speed = edge_speed_kmh(mode_u, flood_depth)
            if speed <= 0: continue

            t_hours = travel_time_hours(rec.length or 0, speed)
            if not np.isfinite(t_hours): continue

            steps_needed = max(1, int(np.ceil(t_hours / step_h)))
            dest_layer = ti + steps_needed

            if dest_layer >= len(time_names): continue

            out.append((
                (uu, ti),
                (vv, dest_layer),
                float(rec.capacity),
                t_hours,
                mode_u,
                float(flood_depth),
            ))

    return out

In [ ]:
t0 = time.time()
G = nx.DiGraph()

# ─────────────────────────────────────
# 1. Add time-layered nodes
# ─────────────────────────────────────
for ti, name in enumerate(time_names):
    for osmid in node_ids:
        row = nodes_unique.loc[nodes_unique["osmid"] == osmid].iloc[0]

        G.add_node(
            (osmid, ti),
            time=name,
            mode=lookups["transport_mode"].loc[osmid],
            is_shelter=bool(row["is_shelter"]),
            is_exit=bool(row.get("is_exit", False)),
            pop=float(lookups["population"].loc[osmid]),
        )

# ─────────────────────────────────────
# 2. Add waiting edges
# ─────────────────────────────────────
for ti in range(len(time_names) - 1):
    step_h = timestep_hour[ti]
    for osmid in node_ids:
        if lookups["shelter_capacity"].loc[osmid]:
            cap = float(lookups["shelter_capacity"].loc[osmid])
        else:
            cap = 1e12

        G.add_edge(
            (osmid, ti),
            (osmid, ti + 1),
            capacity=cap,
            travel_hours=step_h,
            kind="wait",
        )

# ─────────────────────────────────────
# 3. Add movement edges (parallel)
# ─────────────────────────────────────
move_batches = Parallel(n_jobs=cfg.N_JOBS, prefer="threads")(
    delayed(build_move_edges_for_time)(ti)
    for ti in range(len(flood_cols_move))
)

for batch in move_batches:
    for (u_node, v_node, cap, t_hours, mode_u, flood_depth) in batch:
        G.add_edge(
            u_node,
            v_node,
            capacity=cap,
            travel_hours=t_hours,
            kind="move",
            mode=mode_u,
            flood_depth=flood_depth,
        )

print(f"Time-expanded graph built in {time.time()-t0:.1f}s: nodes={G.number_of_nodes()} edges={G.number_of_edges()}")

## Flood Reachability Check

In [ ]:
_shelter_osmids_chk = set(
    int(o) for o in nodes.loc[nodes["is_shelter"] == True, "osmid"].drop_duplicates()
)
_shelter_te_targets = [
    (int(o), layer) for o in _shelter_osmids_chk for layer in range(len(time_names))
]
_G_te_rev = G.reverse(copy=False)
_dist_te = nx.multi_source_dijkstra_path_length(_G_te_rev, _shelter_te_targets, weight="travel_hours")
_reachable_origins = set()
for _nd in _dist_te:
    if isinstance(_nd, tuple) and len(_nd) == 2 and _nd[1] == 0:
        _reachable_origins.add(_nd[0])
_pop_osmids = set(
    int(o) for o in nodes.loc[nodes["pop_total"] > 0, "osmid"].drop_duplicates()
)
_unreachable_flood = _pop_osmids - _reachable_origins
_reach_pop = nodes.loc[
    nodes["osmid"].isin(_reachable_origins) & (nodes["pop_total"] > 0), "pop_total"
].sum()
_unreach_pop = nodes.loc[nodes["osmid"].isin(_unreachable_flood), "pop_total"].sum()
_total_pop = nodes["pop_total"].sum()

if not _unreachable_flood:
    print(f"✅ All {len(_pop_osmids)} populated nodes can reach a shelter in the TEG ({_reach_pop:.0f}/{_total_pop:.0f} people)")
else:
    print(f"⚠️  Flood-unreachable nodes found: {len(_unreachable_flood)}/{len(_pop_osmids)} | {_unreach_pop:.0f}/{_total_pop:.0f} people")
    # Redistribute population from unreachable nodes → nearest reachable neighbor
    _G_static = nx.from_pandas_edgelist(
        edges_use, source="u", target="v", create_using=nx.Graph(), edge_attr="length"
    )
    for _osm in sorted(_unreachable_flood):
        _pop = float(lookups["population"].get(_osm, 0))
        _m = lookups["transport_mode"].get(_osm, "?")
        if _pop <= 0:
            continue
        # BFS on static graph to find nearest reachable neighbor
        _recipient = None
        _visited = {_osm}
        _queue = [_osm]
        while _queue:
            _cur = _queue.pop(0)
            for _nb in _G_static.neighbors(_cur):
                if _nb in _visited:
                    continue
                _visited.add(_nb)
                if _nb in _reachable_origins:
                    _recipient = _nb
                    break
                _queue.append(_nb)
            if _recipient is not None:
                break
        if _recipient is None:
            print(f"     osmid={_osm}  pop={_pop:.0f}  mode={_m}  → NO reachable neighbor (dropped)")
            nodes.loc[nodes["osmid"] == _osm, ["pop_total", "pop_male", "pop_female"]] = 0
        else:
            print(f"     osmid={_osm}  pop={_pop:.0f}  mode={_m}  → moved to osmid={_recipient}")
            # Move population columns
            _src_mask = nodes["osmid"] == _osm
            _dst_mask = nodes["osmid"] == _recipient
            for _col in ["pop_total", "pop_male", "pop_female"]:
                _val = nodes.loc[_src_mask, _col].sum()
                nodes.loc[_dst_mask, _col] = nodes.loc[_dst_mask, _col] + _val
                nodes.loc[_src_mask, _col] = 0
    del _G_static

    # ── Rebuild lookups after redistribution ──
    pop_lookup = nodes.groupby("osmid")["pop_total"].sum().fillna(0)
    _evac_units = pop_lookup / lookups["vehicle_occ"]
    _evac_units = _evac_units.replace([np.inf, -np.inf], 0).fillna(0)
    evac_units_lookup = _evac_units

    _new_total = nodes["pop_total"].sum()
    print(f"   Redistributed → pop_total={_new_total:.0f} (was {_total_pop:.0f})")
    print(f"   All populated nodes now reachable ✅")
    

del _G_te_rev, _dist_te, _reachable_origins, _pop_osmids
del _unreachable_flood, _shelter_te_targets, _shelter_osmids_chk
del _reach_pop, _unreach_pop, _total_pop

## Maximum Flow

In [ ]:
def compute_nearest_shelters(G, nodes, pop_lookup):
    """
    Multi-source Dijkstra from all shelters.
    Assign each node to nearest shelter.

    Returns:
        nearest_shelter: {node_osmid: shelter_osmid}
        pop_per_shelter: aggregated population
    """
    shelters = set(
        int(o) for o in nodes.loc[nodes["is_shelter"] == True, "osmid"].drop_duplicates()
    )
    if not shelters:
        raise ValueError("No shelters found.")

    G_rev = G.reverse(copy=False)
    dist, paths = nx.multi_source_dijkstra(G_rev, sources=[(s, l) for s in shelters for l in range(len(time_names))], weight="travel_hours")

    nearest_shelter = {}
    pop_per_shelter = defaultdict(float)

    for node, _ in dist.items():
        osmid = node[0]

        # Find which shelter this path came from
        path = paths[node]
        source = path[0]  # first node = shelter
        shelter_osmid = source[0]

        if osmid not in nearest_shelter:
            nearest_shelter[osmid] = shelter_osmid

    # Aggregate population
    for osmid, shelter in nearest_shelter.items():
        pop_per_shelter[shelter] += float(pop_lookup.get(osmid, 0))

    return nearest_shelter, dict(pop_per_shelter)

In [ ]:
def build_flow_graph(G):
    """Create flow graph with super source/sink."""
    H = G.copy()
    H.add_node("super_source")
    H.add_node("super_sink")
    return H


def add_source_edges(H, pop_lookup):
    """Connect super source to origin nodes."""
    for osmid, pop in pop_lookup.items():
        if pop > 0:
            H.add_edge("super_source", (int(osmid), 0), capacity=float(pop))


def add_sinks(H, shelters, exits, nodes):
    """
    Add sink edges for shelter nodes and exit nodes.
    Capacity is enforced at the collector → super_sink edge.
    """

    shelter_caps = (
        nodes.loc[nodes["is_shelter"]]
        .groupby("osmid")["shelter_capacity"]
        .max()
        .to_dict()
    )
    
    for osmid in shelters:
        osmid = int(osmid)
        cap = shelter_caps.get(osmid, 0)
        if cap == 0: continue

        collector = f"shelter_{osmid}"
        H.add_node(collector)
        H.add_edge(collector, "super_sink", capacity=float(cap))
        for l in range(len(time_names)):
            H.add_edge((osmid, l), collector, capacity=float(cap))

    for osmid in exits:
        osmid = int(osmid)
        collector = f"exit_{osmid}"
        H.add_node(collector)
        H.add_edge(collector, "super_sink", capacity=1e9)
        for l in range(len(time_names)):
            H.add_edge((osmid, l), collector, capacity=1e9)

Evacuation pipeline

In [ ]:
nearest_shelter_per_nodes, ideal_pop = compute_nearest_shelters(
        G, nodes, lookups["population"]
    )

H = build_flow_graph(G)
add_source_edges(H, lookups["population"])
shelters = nodes.loc[nodes["is_shelter"] == True, "osmid"].unique()
exits = (
    nodes.loc[nodes.get("is_exit", False) == True, "osmid"].unique()
    if "is_exit" in nodes.columns
    else []
)
add_sinks(H, shelters, exits, nodes)

flow_value, flow_dict = nx.maximum_flow(
        H,
        "super_source",
        "super_sink",
        capacity="capacity",
        flow_func=nx.algorithms.flow.edmonds_karp
    )

### Calculating Result

In [ ]:
def compute_evacuation_metrics(flow_dict, pop_lookup, total_people):
    """Compute total evacuated and evacuation rate."""
    super_source = "super_source"

    evac_people = sum(
        flow_dict.get(super_source, {}).get((int(osmid), 0), 0.0)
        for osmid in pop_lookup.index
    )

    evac_rate = evac_people / total_people if total_people > 0 else 0

    return evac_people, evac_rate


def extract_sink_flows(flow_dict, exit_osmids):
    """
    Extract how many people reached each shelter/exit.
    
    MUCH faster than path decomposition.
    """
    from collections import defaultdict

    shelter_flow = defaultdict(float)
    exit_flow = defaultdict(float)

    for u, nbrs in flow_dict.items():
        if not isinstance(u, tuple):
            continue

        osmid, layer = u

        for v, f in nbrs.items():
            if f <= 0:
                continue

            # Case 1: direct to super_sink
            if v == "super_sink":
                if osmid in exits:
                    exit_flow[osmid] += f
                else:
                    shelter_flow[osmid] += f

            # Case 2: via collector
            elif isinstance(v, str):
                if v.startswith("shelter_"):
                    shelter_id = int(v.split("_")[1])
                    shelter_flow[shelter_id] += f
                elif v.startswith("exit_"):
                    exit_id = int(v.split("_")[1])
                    exit_flow[exit_id] += f

    return dict(shelter_flow), dict(exit_flow)


def compute_evacuation_completion(flow_dict, time_stamps):
    """Find last time layer where flow reaches sink."""
    arrival_layers = []

    for u, nbrs in flow_dict.items():
        if not isinstance(u, tuple):
            continue

        for v, f in nbrs.items():
            if f > 0 and (v == "super_sink" or isinstance(v, str)):
                arrival_layers.append(int(u[1]))

    if not arrival_layers:
        return None, None, None

    layer = max(arrival_layers)
    timestamp = time_stamps[layer]
    hours = (timestamp - time_stamps[0]).total_seconds() / 3600.0

    return layer, timestamp, hours



def print_shelter_comparison(ideal_pop, actual_shelter, actual_exit):
    """Compare Dijkstra vs max-flow results."""
    
    all_s = sorted(set(ideal_pop) | set(actual_shelter))

    print(f"\n{'Shelter':>15s} | {'IDEAL':>12s} | {'ACTUAL':>12s} | {'Δ':>8s}")
    print("-" * 55)

    ideal_total = 0
    actual_total = 0

    for s in all_s:
        ideal = ideal_pop.get(s, 0)
        actual = actual_shelter.get(s, 0)

        ideal_total += ideal
        actual_total += actual

        print(f"{s:>15d} | {ideal:>12.0f} | {actual:>12.0f} | {actual - ideal:>+8.0f}")

    for s, val in actual_exit.items():
        actual_total += val
        print(f"{'exit '+str(s):>15s} | {'—':>12s} | {val:>12.0f} |")

    print("-" * 55)
    print(f"{'TOTAL':>15s} | {ideal_total:>12.0f} | {actual_total:>12.0f}")

In [ ]:
total_people = float(lookups["population"].sum())

# --- Metrics ---
evac_people, evac_rate = compute_evacuation_metrics(
    flow_dict, lookups["population"], total_people
)

# --- Sink flows (FAST) ---
shelter_flow_people, exit_flow_people = extract_sink_flows(
    flow_dict, exits
)

# --- Completion time ---
(
    evac_completion_layer,
    evac_completion_timestamp,
    evac_completion_hours,
) = compute_evacuation_completion(flow_dict, time_stamps)

# --- Summary ---
evac_shelter = sum(shelter_flow_people.values())
evac_exit = sum(exit_flow_people.values())

print(f"Total demand: {total_people:.0f}")
print(f"   Evacuated: {evac_people:.0f} ({evac_rate:.1%})")
print(f"     Shelter: {evac_shelter:.0f}")
print(f"     Exit   : {evac_exit:.0f}")

# --- Comparison ---
print_shelter_comparison(
    ideal_pop,
    shelter_flow_people,
    exit_flow_people
)

print(evac_completion_timestamp)
print(evac_completion_hours)

In [ ]:
flow_result = {
    "flow_value": evac_people,
    "flow_units_value": flow_value,
    "evac_rate": evac_rate,
    "flow_dict": flow_dict,
    "graph": H,
    "shelter_flow_people": shelter_flow_people,
    "exit_flow_people": exit_flow_people,
    "exit_osmids": exits,
    "evac_completion_layer": evac_completion_layer,
    "evac_completion_timestamp": evac_completion_timestamp,
    "evac_completion_hours": evac_completion_hours,
    "dijkstra_ideal_pop": ideal_pop,
}

Save the result

In [ ]:
from datetime import datetime
import pickle

# fallback if running manually
if RESULT_FOLDER_NAME is None:
    RESULT_FOLDER_NAME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

folder_path = RESULT_DIR / RESULT_FOLDER_NAME
folder_path.mkdir(parents=True, exist_ok=True)

print(f"Saving results to: {folder_path}")

with open(f"{folder_path}/flow_result.pkl", "wb") as f:
    pickle.dump(flow_result, f)

In [ ]:
timeline_df.to_parquet(folder_path / "flood_simulation_timeline.parquet")

for col in edges.select_dtypes(include='object').columns:
    edges[col] = edges[col].astype(str)

nodes.to_parquet(folder_path / "nodes.parquet")
edges.to_parquet(folder_path / "edges.parquet")

In [ ]:
# Prevent papermill to excute below cells
import sys
sys.exit(0)

In [ ]:
# Quick diagnostics for stranded population and reachability
try:
    col0 = flood_cols_move[0] if flood_cols_move else None
    if col0 and col0 in edges:
        print("flood_depth_t0 value counts:")
        print(edges[col0].value_counts(dropna=False).head())
        # open_ratio = float((edges[col0] < cfg.IMPASSABLE_FLOOD_LEVEL).mean())
        # print(f"Open edge ratio at t0 (< impassable_level={cfg.IMPASSABLE_FLOOD_LEVEL}): {open_ratio:.3f}")
    else:
        print("No t0 flood column found")

    shelter_count = int(nodes.get("is_shelter", pd.Series(dtype=bool)).sum())
    print("Shelters tagged:", shelter_count)

    # Reachability ignoring capacities: reverse BFS from shelters to layer-0 origins
    targets = [
        (int(osm), len(time_names) - 1)
        for osm in nodes.loc[nodes.get("is_shelter", False) == True, "osmid"].drop_duplicates()
    ]
    if not targets:
        print("No shelters found to test reachability.")
    else:
        G_rev = G.reverse(copy=False)
        reachable = set()
        for t in targets:
            reachable.update(nx.descendants(G_rev, t))
            reachable.add(t)
        origins = [
            (int(osm), 0)
            for osm in nodes.loc[nodes.get("pop_total", 0) > 0, "osmid"].drop_duplicates()
        ]
        reachable_origins = [o for o in origins if o in reachable]
        pop_reachable = nodes.loc[
            nodes["osmid"].isin([o[0] for o in reachable_origins]), "pop_total"
        ].sum()
        pop_total_all = nodes.get("pop_total", pd.Series(dtype=float)).sum()
        print(
            f"Reachable origins (layer0): {len(reachable_origins)}/{len(origins)}"
            f" | pop reachable={pop_reachable:.0f}/{pop_total_all:.0f}"
        )
except Exception as exc:
    print("Diagnostics error:", exc)
